# **Data Leakage**

**Prerequisites:** Cross-Validation, Sampling | **Next:** Feature Engineering | **Depth tier:** Foundation

## 1. Theory

Data leakage is when information that wouldn't be available at real
prediction time sneaks into training - producing a model that looks great
on your evaluation but fails in deployment. It is, empirically, one of the
single most common real-world ML bugs - more damaging than most algorithm
choices, and invisible unless specifically checked for.

## 2. The Formal Mechanism

Leakage breaks the i.i.d. train/test separation assumption
(`02_Data/02_sampling.ipynb`) in a specific direction: test-time
information (directly or via a proxy) contaminates training, making the
**empirical risk on the held-out set** (`07_ML_Theory\01_hypothesis_spaces_and_erm.ipynb`) an
overly optimistic estimate of true generalization risk — the evaluation
number becomes wrong, not just the model.

### Categories, each with a concrete example

1. **Target leakage**: a feature that is itself a consequence of the
   target, only available after the outcome is known (e.g. predicting
   hospital readmission using a "discharge follow-up scheduled" column,
   which is only set *because* the hospital already suspected readmission
   risk).

2. **Train/test contamination**: preprocessing (scaling, imputation,
   feature selection) fit on the *full* dataset before splitting — the
   scaler's mean/std then reflects test data statistics, technically
   leaking test-set information into transformations applied to training
   data.

3. **Temporal leakage**: using future information to predict the past —
   e.g. random k-fold splitting on time-series data, where a "future"
   fold trains a model then evaluated on an earlier "past" fold, which
   couldn't have been available at that historical prediction time.
   Direct link to `05_Model_Evaluation/07_validation_strategies.ipynb`'s
   `TimeSeriesSplit` content.

4. **Group leakage**: multiple rows belong to the same real-world entity
   (e.g. multiple scans from the same patient) and a random split puts
   some of that entity's rows in train and some in test — the model can
   partially "recognize" the entity rather than generalizing. Direct link
   to that same file's group-k-fold content.

## 3. Worked Numerical Example - quantifying contamination's optimistic bias

Simulate: true relationship is pure noise (features carry NO real signal
about $y$), but preprocessing computes feature statistics (e.g. mean-
imputation values, or feature selection by correlation with $y$) using
the FULL dataset including the test fold, before splitting.
```
n=200, 50 random noise features, y = random binary label (no real relationship)
```
A model using leaked feature-selection (select the 5 features most
correlated with $y$, computed on the FULL dataset, THEN split) will show
inflated test accuracy — because it selected features that happened to
correlate with the *test* labels too, purely by chance, since selection
saw them. A model doing the correct order (split FIRST, then select
features using only the training fold) should show accuracy near chance
(0.5) — the true, honest answer given the data is pure noise.

## 5. Python - reproduce the leakage numerically

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(0)
n, d = 200, 50
X = rng.normal(size=(n, d))
y = rng.integers(0, 2, size=n)   # pure noise -- X has NO real relationship to y

# LEAKY: select features using ALL data (including future test fold), then split
selector_leaky = SelectKBest(f_classif, k=5).fit(X, y)
X_selected_leaky = selector_leaky.transform(X)
Xtr, Xte, ytr, yte = train_test_split(X_selected_leaky, y, test_size=0.3, random_state=0)
model = LogisticRegression().fit(Xtr, ytr)
print("Leaky pipeline test accuracy:", model.score(Xte, yte))

# CORRECT: split first, select features using ONLY the training fold
Xtr2, Xte2, ytr2, yte2 = train_test_split(X, y, test_size=0.3, random_state=0)
selector_correct = SelectKBest(f_classif, k=5).fit(Xtr2, ytr2)   # fit on train only
Xtr2_sel = selector_correct.transform(Xtr2)
Xte2_sel = selector_correct.transform(Xte2)   # transform test, don't refit
model2 = LogisticRegression().fit(Xtr2_sel, ytr2)
print("Correct pipeline test accuracy:", model2.score(Xte2_sel, yte2))

Leaky pipeline test accuracy: 0.6
Correct pipeline test accuracy: 0.5166666666666667


## 6. Practical Implementation - the general fix

`sklearn.pipeline.Pipeline` — wrap scaling/imputation/feature-selection
and the model together, then cross-validate the *whole pipeline*, so every
fold's preprocessing is refit on only that fold's training data
automatically, structurally preventing category-2 leakage above.